# IEEE-CIS Heterogeneous GNN Benchmark on Colab

Notebook này dùng để benchmark nhánh GNN standalone trên Google Colab trước khi fusion `gnn_score` vào pipeline IEEE chính.

## Runtime

Trong Colab, đặt `Runtime -> Change runtime type -> Hardware accelerator -> GPU` trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Repo download config
GITHUB_REPO = "https://github.com/Thanh-000/FDB1-DoAn.git"
REPO_PARENT = "/content/drive/MyDrive"
REPO_NAME = "FDB1-DoAn"

In [ ]:
import os

repo_path = os.path.join(REPO_PARENT, REPO_NAME)
if not os.path.exists(repo_path):
    !git clone "$GITHUB_REPO" "$repo_path"
else:
    %cd "$repo_path"
    !git pull

print("REPO_DIR:", repo_path)

In [ ]:
# Chỉnh các biến này theo Drive của bạn.
REPO_DIR = "/content/drive/MyDrive/FDB1-DoAn"
ZIP_PATH = "/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip"
EXTRACT_DIR = "/content/ieee-fraud-detection"
DATA_DIR = None

# Benchmark config
FOLD_INDEX = 0
N_SPLITS = 5
EPOCHS = 30
HIDDEN_DIM = 64

In [ ]:
%cd "$REPO_DIR"
!ls gnn

In [ ]:
import os
import zipfile
import glob

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

txn_files = glob.glob(f"{EXTRACT_DIR}/**/train_transaction.csv", recursive=True)
id_files = glob.glob(f"{EXTRACT_DIR}/**/train_identity.csv", recursive=True)

print("txn:", txn_files[:1])
print("id :", id_files[:1])

DATA_DIR = os.path.dirname(txn_files[0])
print("DATA_DIR:", DATA_DIR)
!ls "$DATA_DIR"

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
!pip -q install scikit-learn torch-geometric

In [ ]:
import torch
import torch_geometric
import sklearn
import pandas as pd

print("torch:", torch.__version__)
print("torch_geometric:", torch_geometric.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("CUDA:", torch.cuda.is_available())

## Run one-fold benchmark

Cell dưới sẽ chạy `ieee_gnn_runner.py` với các tham số đã đặt ở phần config.

In [ ]:
import os
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "run_ieee_gnn.py",
    "--data-dir", DATA_DIR,
    "--fold-index", str(FOLD_INDEX),
    "--n-splits", str(N_SPLITS),
    "--epochs", str(EPOCHS),
    "--hidden-dim", str(HIDDEN_DIM),
]

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

## Stronger benchmark

Nếu fold đầu chạy sạch và có tín hiệu, đổi `EPOCHS` và `HIDDEN_DIM` ở cell config rồi chạy lại cell benchmark.

Ví dụ:
- `EPOCHS = 50`
- `HIDDEN_DIM = 96`

## Gửi lại kết quả

Sau khi chạy xong, gửi lại các dòng:

- `[GNN] Train: ...`
- `[GNN] txn_feature_cols (...)`
- `train_auprc`
- `val_auprc`
- `val_roc_auc`

Từ đó sẽ quyết định có fusion `gnn_score` vào notebook IEEE chính hay không.